In [0]:
df=spark.read.format("parquet").load("s3://pipeline-project-uk/bronze/DimArtist")

In [0]:
df.limit(5).display()

In [0]:
df.printSchema()

In [0]:
df.show()

In [0]:
from pyspark.sql.functions import *

In [0]:
df = df.withColumn("artist_name", initcap(trim(col("artist_name")))) \
    .withColumn("genre",       initcap(trim(col("genre")))) \
    .withColumn("country",     trim(col("country"))) \
    .withColumn("updated_at",  to_timestamp(col("updated_at")))

In [0]:
df.limit(5).display()


In [0]:
df = df \
    .withColumn("first_name", split(col("artist_name"), " ")[0]) \
    .withColumn("last_name",  element_at(split(col("artist_name"), " "), -1))

In [0]:
df = df \
    .withColumn("genre_category",
                when(col("genre").isin("Rock", "Pop", "Electronic"),lit("Modern"))
               .when(col("genre").isin("Jazz", "Classical"),lit("Traditional"))
               .when(col("genre") == "Hip-Hop",lit("Urban"))
               .otherwise(lit("Other")))

In [0]:
df = df.withColumn("last_updated_year", year(col("updated_at"))) \
    .withColumn("last_updated_month", month(col("updated_at"))) \
    .withColumn("days_since_last_update", datediff(current_date(), col("updated_at")))

In [0]:
display(df)

In [0]:
df.printSchema()

In [0]:
df = df.select(
    "artist_id",
    "artist_name",
    "genre",
    "country",
    "first_name",
    "last_name",
    "genre_category",
    "last_updated_year",
    "last_updated_month",
    "days_since_last_update",
    "updated_at"                  
)

In [0]:
df.display()